# 00_obtencao_dos_dados
Este notebook obtém (web scraping) os artigos dos anais WIE e WEI do portal SOL/SBC, salva planilhas por ano e gera planilhas consolidadas.

In [ ]:
from pathlib import Path
import sys
import time
import pandas as pd

# Localiza a pasta raiz do projeto: procura README.md e pasta dados
inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
	raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import (normalizar_nome_coluna, 
						   get_edicoes, get_artigos, 
						   get_dados_artigo, consolidar)

# Pastas do projeto (seguindo o padrão existente)
pasta_brutos = raiz / 'dados' / '0_brutos'
pasta_brutos.mkdir(parents=True, exist_ok=True)

### 0.1 Parâmetros de execução

In [ ]:
# Sites a serem raspados e intervalo de anos
SITES = {
	'WIE': 'https://sol.sbc.org.br/index.php/wie/issue/archive',
	'WEI': 'https://sol.sbc.org.br/index.php/wei/issue/archive'
}

YEAR_START = 2007
YEAR_END = 2025
SLEEP_BETWEEN = 0.5

### 0.2 Execução: raspagem por site e por ano

In [ ]:
def run_scrape(save_dir:pasta_brutos.__class__=pasta_brutos, year_start=YEAR_START, year_end=YEAR_END, sleep_between=SLEEP_BETWEEN):
	for site_name, archive_url in SITES.items():
		print(f'🔄️ Iniciando {site_name} -> {archive_url}')
		edicoes = get_edicoes(archive_url)
		for ano in range(year_start, year_end + 1):
			print(f'Processando ano {ano} ({site_name})')
			dados = []
			eds = [e for e in edicoes if e['ano'] == str(ano)]
			if not eds:
				print('Nenhuma edição encontrada para este ano.')
				continue
			for ed in eds:
				print('Edição:', ed['titulo'])
				artigos = get_artigos(ed['url'])
				for art_url in artigos:
					try:
						info = get_dados_artigo(art_url, ano)
						dados.append(info)
						time.sleep(sleep_between)
					except Exception as e:
						print('Erro ao processar', art_url, e)
			if dados:
				df = pd.DataFrame(dados)
				nome_arquivo = save_dir / f'{site_name.lower()}_artigos_{ano}.xlsx'
				df.to_excel(nome_arquivo, index=False)
				print('✅ Salvo:', nome_arquivo, '\n')
			else:
				print('Nenhum artigo coletado para', ano)

# Para rodar, descomente a linha abaixo ou chame run_scrape() manualmente.
# run_scrape()

### 0.3 Consolidação

In [ ]:
def consolidar(site_name, save_dir=pasta_brutos, out_dir=pasta_brutos, year_start=YEAR_START, year_end=YEAR_END):
	out_dir.mkdir(parents=True, exist_ok=True)
	arquivo_saida = out_dir / f'{site_name.lower()}_artigos_consolidados.xlsx'
	with pd.ExcelWriter(arquivo_saida, engine='openpyxl') as writer:
		for ano in range(year_start, year_end + 1):
			nome_arquivo = save_dir / f'{site_name.lower()}_artigos_{ano}.xlsx'
			if nome_arquivo.exists():
				print('Lendo', nome_arquivo.name)
				abas = pd.read_excel(nome_arquivo, sheet_name=None, engine='openpyxl')
				# cada arquivo anual pode conter uma única aba; 
				# se o arquivo tiver múltiplas abas, iteramos
				for aba_nome, df in abas.items():
					df.columns = [normalizar_nome_coluna(c) for c in df.columns]
					sheet_name = str(ano)[:31]
					df.to_excel(writer, sheet_name=sheet_name, index=False)
			else:
				print('Arquivo não encontrado:', nome_arquivo.name)
	print('Consolidado salvo em', arquivo_saida)


# Para rodar, descomente as linhas abaixo ou chame consolidar() manualmente.
# consolidar('WIE')
# consolidar('WEI')